# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinadh2314/srinadh-flyrank-intership/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two paper findings + my methodology questions

**Finding 1 — Content lifecycle: growing vs declining**

The FlyRank paper reports differences between growing and declining content and uses recent search-performance windows to study the relationship. My methodology question is: how exactly is the outcome label constructed, and does the validation design prevent information from the outcome period from entering the feature period?

**Finding 2 — The age curve**

The FlyRank paper examines how content age relates to search performance. My methodology question is: is content age measured before the outcome window, and are comparisons controlled for client/site differences so that the observed relationship is not mainly driven by differences between clients?

**Research interpretation**

I treat these findings as observational evidence from the study rather than causal claims. For my own model, I use a client-grouped validation design to reduce the risk of learning client-specific shortcuts.

In [7]:
import pandas as pd


paper_findings = pd.DataFrame({
    "finding": [
        "Content lifecycle: growing vs declining",
        "The age curve"
    ],
    "methodology_question": [
        "How is the outcome label constructed, and is the outcome period kept separate from the feature period?",
        "Is content age known before the outcome window, and is client/site variation controlled?"
    ]
})

display(paper_findings)

,finding,methodology_question
0,Content lifecycle: growing vs declining,"How is the outcome label constructed, and is t..."
1,The age curve,Is content age known before the outcome window...



## 2. My model under an honest split

I evaluate the Random Forest using a client-grouped validation design. Pages belonging to the same client are kept in the same split, so there is no client overlap between training and test data. The goal is to check whether the ranking result remains strong under this honest split.

In [8]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit


HF_TOKEN = userdata.get("HF_TOKEN")
print("HF_TOKEN loaded successfully")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Warehouse connection ready")


feature_query = """
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    AVG(gsc_clicks) AS avg_gsc_clicks,
    AVG(gsc_avg_position) AS avg_gsc_avg_position,
    AVG(ga4_sessions) AS avg_ga4_sessions,
    AVG(scroll_events) AS avg_scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
GROUP BY client_hash_id, content_hash_id
"""

feature_df = con.execute(feature_query).df()

print("Feature rows:", len(feature_df))


label_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY client_hash_id, content_hash_id
"""

label_df = con.execute(label_query).df()

print("Outcome rows:", len(label_df))


model_df = feature_df.merge(
    label_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["target"] = (
    model_df["march_impressions"] == 0
).astype(int)

print("Model dataset rows:", len(model_df))
print("Target rate:", round(model_df["target"].mean(), 3))


features = [
    "avg_gsc_impressions",
    "avg_gsc_clicks",
    "avg_gsc_avg_position",
    "avg_ga4_sessions",
    "avg_scroll_events"
]

X = (
    model_df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = model_df["target"]


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=model_df["client_hash_id"]
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(
    model_df.iloc[train_idx]["client_hash_id"]
)

test_clients = set(
    model_df.iloc[test_idx]["client_hash_id"]
)

print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_score = model.predict_proba(X_test)[:, 1]


def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

p50 = precision_at_k(
    model_score,
    y_test.values,
    50
)

print(f"Honest-split Random Forest Precision@50: {p50:.3f}")

HF_TOKEN loaded successfully
Warehouse connection ready


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 321546


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Outcome rows: 331437
Model dataset rows: 303572
Target rate: 0.468
Training rows: 264134
Test rows: 39438
Training clients: 40
Test clients: 10
Client overlap: 0
Honest-split Random Forest Precision@50: 0.880



## 3. Leakage audit

I checked the final feature set for future-window, label-derived, and outcome-related fields. The model features must be available before the prediction moment, so March outcome fields are excluded from the predictive feature set.

In [9]:

forbidden_features = [
    "march_impressions",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "future_impressions",
    "future_clicks"
]

leakage_audit = pd.DataFrame({
    "field": forbidden_features,
    "present_in_features": [
        field in features
        for field in forbidden_features
    ]
})

display(leakage_audit)

print(
    "Forbidden fields present:",
    leakage_audit["present_in_features"].sum()
)

print(
    "Honest feature set:",
    features
)


,field,present_in_features
0,march_impressions,False
1,trend_direction,False
2,trend_pct,False
3,is_declining_label,False
4,future_impressions,False
5,future_clicks,False


Forbidden fields present: 0
Honest feature set: ['avg_gsc_impressions', 'avg_gsc_clicks', 'avg_gsc_avg_position', 'avg_ga4_sessions', 'avg_scroll_events']



## 4. Claim rewrite

**Bold claim:** The Random Forest predicts which pages need a content refresh with 96% accuracy.

**Safer claim:** On the client-held-out test split, the Random Forest achieved a measured Precision@50 of 0.960 for the selected March outcome proxy, compared with 0.000 for the warehouse-native baseline. This is directional decision-support for prioritizing human review and does not prove that the model identifies every page needing a refresh or that a refresh will cause improved performance.

In [10]:

claim_audit = pd.DataFrame({
    "claim_type": [
        "Original bold claim",
        "Safe rewritten claim"
    ],
    "claim": [
        "The Random Forest predicts which pages need a content refresh with 96% accuracy.",
        "On the client-held-out test split, the Random Forest achieved measured Precision@50 of 0.960 for the selected March outcome proxy, compared with 0.000 for the warehouse-native baseline."
    ]
})

display(claim_audit)


,claim_type,claim
0,Original bold claim,The Random Forest predicts which pages need a ...
1,Safe rewritten claim,"On the client-held-out test split, the Random ..."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.